# Notebook 10 — Reentrenamiento HemoVet v4

**Fase 3 · Módulo 3F** — Incorporación de 1,213 hemogramas caninos válidos extraídos de 1,279 archivos MYTHIC_VET (`Sin Limpieza/RESULTS/`) para reducir FN, con ajuste balanceado y promoción condicionada.

**Entrada:** `Sin Limpieza/RESULTS/**/*.CSV`, `data/processed/train_augmented.csv`, `data/processed/val_augmented.csv`, `models/best_model_v3.pkl`  
**Salida:** `data/processed/results_augmented.csv`, `models/best_model_v4.pkl`, `data/processed/decision_thresholds_v4.json`

| Sección | Contenido |
|---|---|
| **1** | Setup y configuración |
| **2** | Parser MYTHIC_VET CSV |
| **3** | Extracción CBC de todos los RESULTS |
| **4** | Feature engineering |
| **5** | Etiquetado desde flags interpretativos |
| **6** | Verificación de distribución |
| **7** | Reentrenamiento XGBoost v4 |
| **8** | Optimización de thresholds (F-beta β=2) |
| **9** | Comparación v3 vs v4 |

### Guia rapida del codigo

- Lee nuevos CSV MYTHIC_VET y los convierte al esquema CBC del proyecto.
- Construye features, etiqueta desde flags interpretativos y arma un dataset aumentado.
- Reentrena XGBoost v4 y optimiza thresholds con F-beta.
- Compara v3 vs v4 sin reemplazar automaticamente el modelo oficial.


## 1. Setup

In [1]:
import re
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

_cwd = Path.cwd()
PROJECT = _cwd
while PROJECT.name != 'hemogramas-proyectoICC' and PROJECT.parent != PROJECT:
    PROJECT = PROJECT.parent
if PROJECT.name != 'hemogramas-proyectoICC':
    raise RuntimeError(f'No se localizó la raiz del proyecto. CWD: {_cwd}')

DATA_PROC = PROJECT / 'data' / 'processed'
MODELS    = PROJECT / 'models'
RESULTS   = PROJECT / 'Sin Limpieza' / 'RESULTS'

with open(DATA_PROC / 'feature_columns.json') as f:
    FEATURES = json.load(f)['feature_columns']

OFFICIAL_LABELS = [
    'QC_REQUIERE_FROTIS',
    'PATRON_INFLAMATORIO',
    'PATRON_LEUCOGRAMA_ESTRES',
    'PATRON_ANEMIA_NO_REGENERATIVA',
    'PATRON_HEMOLISIS_MCHC',
    'PATRON_POLICITEMIA',
    'PATRON_ANEMIA_REGENERATIVA',
]

print(f'Raiz del proyecto: {PROJECT}')
print(f'Features: {len(FEATURES)}')
print(f'Labels: {OFFICIAL_LABELS}')

Raiz del proyecto: /home/edwinb/tesis/hemogramas-proyectoICC
Features: 43
Labels: ['QC_REQUIERE_FROTIS', 'PATRON_INFLAMATORIO', 'PATRON_LEUCOGRAMA_ESTRES', 'PATRON_ANEMIA_NO_REGENERATIVA', 'PATRON_HEMOLISIS_MCHC', 'PATRON_POLICITEMIA', 'PATRON_ANEMIA_REGENERATIVA']


## 2. Parser MYTHIC_VET CSV

In [2]:
# Mapping MYTHIC_VET param names → canonical names usados en el modelo
_PARAM_MAP = {
    'WBC':   'WBC',
    'RBC':   'RBC',
    'HGB':   'HGB',
    'HCT':   'HCT',
    'PLT':   'Platelets',
    'LYM':   'Lymphocytes',
    'MON':   'Monocytes',
    'GRA':   'Neutrophils',
    r'LYM%': 'Lymphocytes_pct',
    r'MON%': 'Monocytes_pct',
    r'GRA%': 'Neutrophils_pct',
    'MCV':   'MCV',
    'MCH':   'MCH',
    'MCHC':  'MCHC',
    'RDW':   'RDW',
    'MPV':   'MPV',
}


def parse_mythic_vet(filepath: Path) -> dict | None:
    """
    Parsea un CSV MYTHIC_VET y retorna un dict con CBC numérico + set de flags interpretativos.
    Retorna None si el archivo no es reconocido como hemograma canino (TYPE != DOG).
    """
    try:
        with open(filepath, 'r', encoding='latin-1', errors='replace') as f:
            content = f.read().replace('\r', '').replace('\n', '')
    except Exception:
        return None

    # Solo procesar hemogramas caninos
    if 'TYPE;DOG' not in content:
        return None

    result = {}

    # Extraer valores CBC numéricos
    for raw_name, canonical in _PARAM_MAP.items():
        pattern = rf'{re.escape(raw_name)};(\d+(?:\.\d+)?)\s*;'
        m = re.search(pattern, content)
        result[canonical] = float(m.group(1)) if m else None

    # Extraer flags interpretativos
    flags = set()
    interp_m = re.search(
        r'INTERPRETIVE_WBC;(.*?)INTERPRETIVE_RBC;(.*?)INTERPRETIVE_PLT;(.*?)(?:END_RESULT|$)',
        content
    )
    if interp_m:
        for group in interp_m.groups():
            for flag in group.split(';'):
                f = flag.strip()
                if f and f != 'NO_INTERPRETATION':
                    flags.add(f)

    result['interpretive_flags'] = flags
    result['source_file'] = str(filepath.relative_to(PROJECT))
    return result


# Prueba con un archivo de muestra
sample_files = list(RESULTS.glob('**/*.CSV'))[:2]
for sf in sample_files:
    parsed = parse_mythic_vet(sf)
    if parsed:
        print(f'Archivo: {sf.name}')
        print(f'  WBC={parsed.get("WBC")}, HCT={parsed.get("HCT")}, MCHC={parsed.get("MCHC")}')
        print(f'  Flags: {parsed.get("interpretive_flags")}')
        print()

Archivo: 0002.CSV
  WBC=18.0, HCT=6.8, MCHC=22.1
  Flags: {'HYPOCR', 'GRA>', 'MACRO', 'ANE', 'LEU>', 'ANIS>'}

Archivo: 0006.CSV
  WBC=26.6, HCT=53.6, MCHC=31.9
  Flags: {'GRA>', 'LYM>', 'LEU>', 'MON>', 'THR>'}



## 3. Extracción CBC — todos los RESULTS

In [3]:
all_csv_files = sorted(RESULTS.glob('**/*.CSV'))
print(f'Archivos CSV encontrados: {len(all_csv_files)}')

records = []
skipped = 0

for fp in all_csv_files:
    parsed = parse_mythic_vet(fp)
    if parsed is None:
        skipped += 1
        continue
    records.append(parsed)

df_raw = pd.DataFrame(records)
print(f'Registros parseados: {len(df_raw)}')
print(f'Skipped (no TYPE;DOG o error): {skipped}')

# Verificar completitud de campos clave
key_cols = ['WBC', 'RBC', 'HGB', 'HCT', 'Platelets', 'Lymphocytes', 'Neutrophils', 'MCHC', 'MCV']
print('\nCompletitud de campos clave:')
for col in key_cols:
    pct = df_raw[col].notna().mean() * 100
    print(f'  {col:<20} {pct:.1f}%')

print(f'\nHCT stats:')
print(df_raw['HCT'].describe())
print(f'HCT > 55: {(df_raw["HCT"] > 55).sum()}')
print(f'HCT > 60: {(df_raw["HCT"] > 60).sum()}')

df_raw.to_csv(DATA_PROC / 'results_raw.csv', index=False)
print(f'\nGuardado: {DATA_PROC / "results_raw.csv"}')

Archivos CSV encontrados: 1279


Registros parseados: 1213
Skipped (no TYPE;DOG o error): 66

Completitud de campos clave:
  WBC                  100.0%
  RBC                  99.9%
  HGB                  99.8%
  HCT                  98.8%
  Platelets            100.0%
  Lymphocytes          96.9%
  Neutrophils          96.9%
  MCHC                 98.6%
  MCV                  98.9%

HCT stats:
count    1199.000000
mean       39.225438
std        14.214587
min         4.200000
25%        30.100000
50%        38.700000
75%        47.800000
max        85.500000
Name: HCT, dtype: float64
HCT > 55: 160
HCT > 60: 92

Guardado: /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/results_raw.csv


## 4. Feature Engineering

In [4]:
med = pd.read_csv(DATA_PROC / 'imputer_medians_augmented.csv', index_col=0)['mediana_train']


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # --- Campos no disponibles en MYTHIC_VET → imputar con mediana de entrenamiento ---
    for col in ['Eosinophils', 'Basophils', 'PDW']:
        d[col] = med.get(col, 0.0)

    # Reticulocitos: no medidos → imputar a 0 (misma estrategia que val_augmented)
    d['Reticulocytes']     = 0.0
    d['Reticulocytes_pct'] = 0.0

    # Edad: no disponible → mediana
    d['age_years'] = med.get('age_years', 5.0)

    # --- Flags clínicos (mismos umbrales que NB03/build_clinical_flags) ---
    d['flag_thrombocytopenia']        = (d['Platelets'] < 200).astype(int)
    d['flag_thrombocytopenia_severe'] = (d['Platelets'] < 100).astype(int)
    d['flag_lymphopenia']             = (d['Lymphocytes'] < 1.0).astype(int)
    d['flag_lymphocytosis']           = (d['Lymphocytes'] > 5.0).astype(int)
    d['flag_neutrophilia']            = (d['Neutrophils'] > 11.0).astype(int)
    d['flag_neutropenia']             = (d['Neutrophils'] < 2.9).astype(int)
    d['flag_leukocytosis']            = (d['WBC'] > 17.0).astype(int)
    d['flag_leukopenia']              = (d['WBC'] < 5.5).astype(int)

    # --- Flags de serie roja (NB03/build_anemia_index) ---
    d['flag_anemia']     = (d['HGB'] < 12.0).astype(int)
    d['flag_microcytic'] = (d['MCV'] < 62.0).astype(int)
    d['flag_macrocytic'] = (d['MCV'] > 82.0).astype(int)
    d['flag_hypochromic']= (d['MCHC'] < 31.0).astype(int)

    # --- Plaquetario (NB03/build_platelet_features) ---
    d['flag_macro_platelets'] = (d['MPV'] > 12.5).astype(int)

    # --- Ratios (NB03/build_ratio_features) ---
    EPS = 0.01
    d['ratio_NLR']     = d['Neutrophils'] / (d['Lymphocytes'] + EPS)
    d['ratio_PLR']     = d['Platelets']   / (d['Lymphocytes'] + EPS)
    d['ratio_MLR']     = d['Monocytes']   / (d['Lymphocytes'] + EPS)
    d['ratio_MPV_PLT'] = d['MPV']         / (d['Platelets']   + EPS)

    # --- Leucograma de estrés (NB03/build_stress_leukogram) ---
    d['pattern_stress_leukogram'] = (
        (d['Neutrophils'] > 11.0) &
        (d['Lymphocytes'] < 1.0) &
        (d['Eosinophils'] < 0.5)
    ).astype(int)

    # --- Reticulocitos flags → 0 (sin datos) ---
    d['flag_reticulocytosis_pct_gt2']    = 0
    d['flag_reticulocytopenia_abs_lt10'] = 0
    d['flag_regen_anemia_proxy']         = 0

    return d


df_feat = build_features(df_raw)

missing_feats = [f for f in FEATURES if f not in df_feat.columns]
print(f'Features faltantes: {missing_feats}')
print(f'Features disponibles: {len([f for f in FEATURES if f in df_feat.columns])} / {len(FEATURES)}')

Features faltantes: []
Features disponibles: 43 / 43


## 5. Etiquetado desde flags interpretativos

In [5]:
def label_from_flags(row: pd.Series) -> dict:
    """
    Deriva etiquetas PATRON_* desde flags interpretativos MYTHIC_VET y valores CBC.

    Limitaciones documentadas:
    - PATRON_ANEMIA_NO_REGENERATIVA: proxy (ANE sin MACRO), no distingue regenerativa
      sin reticulocitos. Casos con MACRO se excluyen para reducir ruido.
    - PATRON_ANEMIA_REGENERATIVA: 0 (sin reticulocitos, imposible determinar).
    - Umbrales derivados de rangos de referencia MYTHIC_VET para perros.
    """
    flags = row.get('interpretive_flags', set())
    if not isinstance(flags, set):
        # Puede llegar como string al leer desde CSV
        try:
            flags = eval(str(flags))
        except Exception:
            flags = set()

    hct  = row.get('HCT',  None)
    mchc = row.get('MCHC', None)

    return {
        # Policitemia: HCT ≥ 56 (umbral clínico observado en validación S1–S3)
        'PATRON_POLICITEMIA':           int((hct  or 0) >= 56),
        # Leucograma de estrés: granulocitosis + linfopenia simultáneas
        'PATRON_LEUCOGRAMA_ESTRES':     int('GRA>' in flags and 'LYM<' in flags),
        # Hemólisis: MCHC > 36 g/dL (artefacto lipemia/hemólisis in vitro)
        'PATRON_HEMOLISIS_MCHC':        int((mchc or 0) > 36),
        # Inflamatorio: monocitosis (proxy de inflamación crónica)
        'PATRON_INFLAMATORIO':          int('MON>' in flags),
        # Anemia no regenerativa: anemia sin señal macrocítica (regen)
        'PATRON_ANEMIA_NO_REGENERATIVA': int('ANE' in flags and 'MACRO' not in flags),
        # Anemia regenerativa: imposible sin reticulocitos → 0
        'PATRON_ANEMIA_REGENERATIVA':   0,
        # QC frotis: anisocitosis, macrocitosis o macroplaquetas
        'QC_REQUIERE_FROTIS':           int('ANIS>' in flags or 'MACRO' in flags or 'MACROP' in flags),
        # QC agregados: aglutinación en frío
        'QC_AGREGADOS_PLAQUETARIOS':    int('COLDAGG' in flags),
    }


label_dicts = df_feat.apply(label_from_flags, axis=1)
df_labels = pd.DataFrame(label_dicts.tolist())

df_labeled = pd.concat([df_feat, df_labels], axis=1)

print('Distribución de etiquetas en datos RESULTS:')
print(df_labels.sum().to_string())
print(f'\nTotal: {len(df_labeled)} registros')

df_labeled.to_csv(DATA_PROC / 'results_labeled.csv', index=False)
print(f'Guardado: {DATA_PROC / "results_labeled.csv"}')

Distribución de etiquetas en datos RESULTS:
PATRON_POLICITEMIA               144
PATRON_LEUCOGRAMA_ESTRES         176
PATRON_HEMOLISIS_MCHC            111
PATRON_INFLAMATORIO              402
PATRON_ANEMIA_NO_REGENERATIVA    515
PATRON_ANEMIA_REGENERATIVA         0
QC_REQUIERE_FROTIS               502
QC_AGREGADOS_PLAQUETARIOS        117

Total: 1213 registros


Guardado: /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/results_labeled.csv


## 6. Verificación de distribución

In [6]:
train_base = pd.read_csv(DATA_PROC / 'train.csv')

print('=== Comparación de distribuciones train.csv vs RESULTS ===')
print(f'\n{"Etiqueta":<42} {"Train_base":>12} {"RESULTS":>10} {"Total v4":>10}')
print('-' * 78)
for label in OFFICIAL_LABELS:
    n_train  = train_base[label].sum() if label in train_base.columns else 0
    n_res    = df_labels[label].sum()  if label in df_labels.columns  else 0
    print(f'  {label:<40} {n_train:>12} {n_res:>10} {n_train+n_res:>10}')

print()
print('HCT en RESULTS (posibles policitemia):')
hct_vals = df_labeled['HCT'].dropna()
print(f'  Total HCT disponible: {len(hct_vals)}')
print(f'  HCT ≥ 56: {(hct_vals >= 56).sum()}')
print(f'  HCT ≥ 60: {(hct_vals >= 60).sum()}')
print(f'  HCT ≥ 65: {(hct_vals >= 65).sum()}')

=== Comparación de distribuciones train.csv vs RESULTS ===

Etiqueta                                     Train_base    RESULTS   Total v4
------------------------------------------------------------------------------
  QC_REQUIERE_FROTIS                                601        502       1103
  PATRON_INFLAMATORIO                               770        402       1172
  PATRON_LEUCOGRAMA_ESTRES                          769        176        945
  PATRON_ANEMIA_NO_REGENERATIVA                     163        515        678
  PATRON_HEMOLISIS_MCHC                             217        111        328
  PATRON_POLICITEMIA                                109        144        253
  PATRON_ANEMIA_REGENERATIVA                         87          0         87

HCT en RESULTS (posibles policitemia):
  Total HCT disponible: 1199
  HCT ≥ 56: 144
  HCT ≥ 60: 92
  HCT ≥ 65: 43


In [7]:
# Construir results_augmented.csv con solo los 43 features + labels
# Imputar NaN restantes con mediana de entrenamiento
df_aug = df_labeled.copy()

for feat in FEATURES:
    if feat not in df_aug.columns:
        df_aug[feat] = 0.0
    elif df_aug[feat].isna().any():
        df_aug[feat] = df_aug[feat].fillna(med.get(feat, 0.0))

EXPORT_COLS = FEATURES + OFFICIAL_LABELS + ['QC_AGREGADOS_PLAQUETARIOS']
df_aug = df_aug[[c for c in EXPORT_COLS if c in df_aug.columns]]

df_aug.to_csv(DATA_PROC / 'results_augmented.csv', index=False)
print(f'results_augmented.csv: {df_aug.shape}')
print(f'Guardado: {DATA_PROC / "results_augmented.csv"}')

results_augmented.csv: (1213, 51)
Guardado: /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/results_augmented.csv


## 7. Reentrenamiento XGBoost v4

In [8]:
# Usar train.csv (con reticulocitos reales) como base — misma fuente que v3
# Los datos RESULTS tienen reticulocitos=0 (no medidos), lo que el modelo aprende como caso válido
train_base = pd.read_csv(DATA_PROC / 'train.csv')
val_df     = pd.read_csv(DATA_PROC / 'val.csv')

# Asegurar que todos los 43 features están presentes en val
for feat in FEATURES:
    if feat not in val_df.columns:
        val_df[feat] = 0.0

# Combinar train.csv + RESULTS nuevos
results_aug = pd.read_csv(DATA_PROC / 'results_augmented.csv')

for feat in FEATURES:
    if feat not in results_aug.columns:
        results_aug[feat] = 0.0

df_train_v4 = pd.concat(
    [train_base[FEATURES + OFFICIAL_LABELS], results_aug[FEATURES + OFFICIAL_LABELS]],
    ignore_index=True
)

X_train = df_train_v4[FEATURES].values
Y_train = df_train_v4[OFFICIAL_LABELS].values

X_val = val_df[FEATURES].values
Y_val = val_df[OFFICIAL_LABELS].values

print(f'Train v4: {X_train.shape}  (base train.csv: {len(train_base)}, nuevos RESULTS: {len(results_aug)})')
print(f'Val:      {X_val.shape}')
print()
print('Distribución de labels en train v4:')
for i, label in enumerate(OFFICIAL_LABELS):
    n_base = train_base[label].sum() if label in train_base.columns else 0
    n_new  = results_aug[label].sum() if label in results_aug.columns else 0
    print(f'  {label:<42} base={n_base:>5}  new={n_new:>5}  total={n_base+n_new:>5}')

Train v4: (2930, 43)  (base train.csv: 1717, nuevos RESULTS: 1213)
Val:      (368, 43)

Distribución de labels en train v4:
  QC_REQUIERE_FROTIS                         base=  601  new=  502  total= 1103
  PATRON_INFLAMATORIO                        base=  770  new=  402  total= 1172
  PATRON_LEUCOGRAMA_ESTRES                   base=  769  new=  176  total=  945
  PATRON_ANEMIA_NO_REGENERATIVA              base=  163  new=  515  total=  678
  PATRON_HEMOLISIS_MCHC                      base=  217  new=  111  total=  328
  PATRON_POLICITEMIA                         base=  109  new=  144  total=  253
  PATRON_ANEMIA_REGENERATIVA                 base=   87  new=    0  total=   87


In [9]:
def get_scale_pos_weight(Y, label_idx):
    pos = Y[:, label_idx].sum()
    neg = len(Y) - pos
    return neg / pos if pos > 0 else 1.0


# Hiperparámetros idénticos a v3 (best_xgb_params del grid search)
XGB_PARAMS = {
    'n_estimators':     300,
    'max_depth':        8,
    'learning_rate':    0.03,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     42,
    'eval_metric':      'aucpr',
    'verbosity':        0,
}

models_v4 = {}

print('Entrenando XGBoost v4 por etiqueta...')
for i, label in enumerate(OFFICIAL_LABELS):
    spw = get_scale_pos_weight(Y_train, i)
    clf = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=spw)
    clf.fit(
        X_train, Y_train[:, i],
        eval_set=[(X_val, Y_val[:, i])],
        verbose=False,
    )
    models_v4[label] = clf
    print(f'  {label:<42} spw={spw:.2f}')

with open(MODELS / 'best_model_v4.pkl', 'wb') as f:
    pickle.dump(models_v4, f)
print(f'\nGuardado: {MODELS / "best_model_v4.pkl"}')

Entrenando XGBoost v4 por etiqueta...


  QC_REQUIERE_FROTIS                         spw=1.66


  PATRON_INFLAMATORIO                        spw=1.50


  PATRON_LEUCOGRAMA_ESTRES                   spw=2.10


  PATRON_ANEMIA_NO_REGENERATIVA              spw=3.32


  PATRON_HEMOLISIS_MCHC                      spw=7.93


  PATRON_POLICITEMIA                         spw=10.58


  PATRON_ANEMIA_REGENERATIVA                 spw=32.68

Guardado: /home/edwinb/tesis/hemogramas-proyectoICC/models/best_model_v4.pkl


## 8. Calibración y umbrales con restricciones FP/FN

In [10]:
def _logit(prob):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(prob / (1 - prob))


def _apply_sigmoid(prob, coefficient, intercept):
    score = coefficient * _logit(prob) + intercept
    return 1 / (1 + np.exp(-np.clip(score, -40, 40)))


def fit_oof_sigmoid(y_true, raw_prob):
    """Retorna probabilidades OOF y parámetros finales sin tocar el test."""
    positives = int(np.sum(y_true))
    negatives = int(len(y_true) - positives)
    if min(positives, negatives) < 20:
        return np.asarray(raw_prob), None

    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y_true), dtype=float)
    for train_idx, holdout_idx in splitter.split(raw_prob, y_true):
        calibrator = LogisticRegression(C=1.0, solver='lbfgs', random_state=42)
        calibrator.fit(_logit(raw_prob[train_idx]).reshape(-1, 1), y_true[train_idx])
        oof[holdout_idx] = calibrator.predict_proba(
            _logit(raw_prob[holdout_idx]).reshape(-1, 1)
        )[:, 1]

    final_calibrator = LogisticRegression(C=1.0, solver='lbfgs', random_state=42)
    final_calibrator.fit(_logit(raw_prob).reshape(-1, 1), y_true)
    params = {
        'type': 'sigmoid_logit',
        'coefficient': float(final_calibrator.coef_[0, 0]),
        'intercept': float(final_calibrator.intercept_[0]),
    }
    return oof, params


def select_balanced_threshold(y_true, probability, baseline_probability, baseline_threshold):
    """Minimiza FP con pérdida de recall <=2 pp y F1 no inferior."""
    baseline_pred = baseline_probability >= baseline_threshold
    baseline_recall = recall_score(y_true, baseline_pred, zero_division=0)
    baseline_f1 = f1_score(y_true, baseline_pred, zero_division=0)
    candidates = np.unique(np.r_[np.linspace(0.01, 0.99, 197), probability])
    feasible = []
    for threshold in candidates:
        prediction = probability >= threshold
        tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
        recall = recall_score(y_true, prediction, zero_division=0)
        f1 = f1_score(y_true, prediction, zero_division=0)
        if recall >= baseline_recall - 0.02 and f1 >= baseline_f1:
            feasible.append((int(fp), -f1, -recall, float(threshold)))
    return min(feasible)[3] if feasible else float(baseline_threshold)


val_probs_v4_raw = np.column_stack([
    models_v4[label].predict_proba(X_val)[:, 1] for label in OFFICIAL_LABELS
])
with open(DATA_PROC / 'decision_thresholds_v2.json') as f:
    runtime_thresholds = json.load(f)['thresholds']

val_probs_v4_candidate = val_probs_v4_raw.copy()
candidate_calibrators = {}
candidate_thresholds = {}
threshold_rows = []

for i, label in enumerate(OFFICIAL_LABELS):
    y_true = Y_val[:, i].astype(int)
    oof_probability, params = fit_oof_sigmoid(y_true, val_probs_v4_raw[:, i])
    if params is not None:
        candidate_calibrators[label] = params
    candidate_thresholds[label] = select_balanced_threshold(
        y_true, oof_probability, val_probs_v4_raw[:, i], runtime_thresholds[label]
    )
    threshold_rows.append({
        'label': label,
        'support_val': int(y_true.sum()),
        'threshold_runtime': runtime_thresholds[label],
        'threshold_candidate': candidate_thresholds[label],
        'calibrated': params is not None,
        'status': 'exploratory_low_support' if y_true.sum() < 20 else 'eligible',
    })

df_thresholds = pd.DataFrame(threshold_rows)
display(df_thresholds)
print('Los umbrales candidatos quedan congelados antes de abrir test.')

,label,support_val,threshold_runtime,threshold_candidate,calibrated,status
0,QC_REQUIERE_FROTIS,130,0.300000,0.280,True,eligible
1,PATRON_INFLAMATORIO,175,0.641006,0.430,True,eligible
2,PATRON_LEUCOGRAMA_ESTRES,167,0.300000,0.630,True,eligible
3,PATRON_ANEMIA_NO_REGENERATIVA,47,0.455682,0.410,True,eligible
4,PATRON_HEMOLISIS_MCHC,80,0.714316,0.460,True,eligible
5,PATRON_POLICITEMIA,27,0.900000,0.010,True,eligible
6,PATRON_ANEMIA_REGENERATIVA,15,0.478863,0.415,False,exploratory_low_support


Los umbrales candidatos quedan congelados antes de abrir test.


## 9. Comparación v3 vs v4

In [11]:
def evaluate_version(version, split_name, frame, probabilities, thresholds):
    rows = []
    for i, label in enumerate(OFFICIAL_LABELS):
        y_true = frame[label].to_numpy(dtype=int)
        prediction = (probabilities[:, i] >= thresholds[label]).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
        rows.append({
            'version': version, 'split': split_name, 'label': label,
            'n_pos': int(y_true.sum()), 'n_neg': int(len(y_true) - y_true.sum()),
            'thr': thresholds[label],
            'prauc': average_precision_score(y_true, probabilities[:, i]),
            'brier': brier_score_loss(y_true, probabilities[:, i]),
            'f1': f1_score(y_true, prediction, zero_division=0),
            'prec': precision_score(y_true, prediction, zero_division=0),
            'rec': recall_score(y_true, prediction, zero_division=0),
            'spec': tn / (tn + fp) if tn + fp else np.nan,
            'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
        })
    return pd.DataFrame(rows)


test_df = pd.read_csv(DATA_PROC / 'test.csv')
X_test = test_df[FEATURES].values
with open(MODELS / 'best_model_v3.pkl', 'rb') as f:
    models_v3 = pickle.load(f)
with open(DATA_PROC / 'decision_thresholds_v3.json') as f:
    thresholds_v3 = json.load(f)['thresholds']

test_probs_v3 = np.column_stack([
    models_v3[label].predict_proba(X_test)[:, 1] for label in OFFICIAL_LABELS
])
test_probs_v4_raw = np.column_stack([
    models_v4[label].predict_proba(X_test)[:, 1] for label in OFFICIAL_LABELS
])
test_probs_candidate = test_probs_v4_raw.copy()
for i, label in enumerate(OFFICIAL_LABELS):
    params = candidate_calibrators.get(label)
    if params is not None:
        test_probs_candidate[:, i] = _apply_sigmoid(
            test_probs_v4_raw[:, i], params['coefficient'], params['intercept']
        )

df_v3_test = evaluate_version('v3', 'test', test_df, test_probs_v3, thresholds_v3)
df_v4_test = evaluate_version('v4_runtime', 'test', test_df, test_probs_v4_raw, runtime_thresholds)
df_candidate_test = evaluate_version(
    'v4_balanced_candidate', 'test', test_df, test_probs_candidate, candidate_thresholds
)
df_compare = pd.concat([df_v3_test, df_v4_test, df_candidate_test], ignore_index=True)

summary = df_compare.groupby('version').agg(
    prauc_macro=('prauc', 'mean'), brier_macro=('brier', 'mean'),
    f1_macro=('f1', 'mean'), recall_macro=('rec', 'mean'),
    tp=('tp', 'sum'), fp=('fp', 'sum'), fn=('fn', 'sum'), tn=('tn', 'sum'),
)
display(summary)
display(df_compare[['version', 'label', 'n_pos', 'thr', 'tp', 'fp', 'fn', 'prec', 'rec', 'f1', 'prauc']])

runtime_by_label = df_v4_test.set_index('label')
candidate_by_label = df_candidate_test.set_index('label')
recall_constraint = bool((candidate_by_label['rec'] >= runtime_by_label['rec'] - 0.02).all())
f1_constraint = bool((candidate_by_label['f1'] >= runtime_by_label['f1']).all())
fp_constraint = int(df_candidate_test.fp.sum()) < int(df_v4_test.fp.sum())
fn_constraint = int(df_candidate_test.fn.sum()) < int(df_v3_test.fn.sum())
promote_candidate = recall_constraint and f1_constraint and fp_constraint and fn_constraint

print({
    'recall_loss_le_2pp': recall_constraint,
    'f1_not_lower_per_label': f1_constraint,
    'fp_lower_than_v4': fp_constraint,
    'fn_lower_than_v3': fn_constraint,
    'promote_candidate': promote_candidate,
})

comparison_path = PROJECT / 'outputs' / 'comparacion_modelos.csv'
existing = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if not existing.empty and 'version' in existing.columns:
    existing = existing[existing['version'] != 'v4_balanced_candidate']
pd.concat([existing, df_candidate_test], ignore_index=True).to_csv(comparison_path, index=False)
print(f'Guardado: {comparison_path}')

,prauc_macro,brier_macro,f1_macro,recall_macro,tp,fp,fn,tn
version,,,,,,,,
v3,0.957695,0.028219,0.861130,0.886053,567,34,67,1915
v4_balanced_candidate,0.952924,0.029941,0.866914,0.923893,584,56,50,1893
v4_runtime,0.952924,0.029814,0.872674,0.920542,585,54,49,1895


,version,label,n_pos,thr,tp,fp,fn,prec,rec,f1,prauc
0,v3,QC_REQUIERE_FROTIS,136,0.636237,80,9,56,0.898876,0.588235,0.711111,0.860523
1,v3,PATRON_INFLAMATORIO,198,0.546658,197,4,1,0.980100,0.994949,0.987469,0.993609
2,v3,PATRON_LEUCOGRAMA_ESTRES,172,0.390740,171,9,1,0.950000,0.994186,0.971591,0.986072
3,v3,PATRON_ANEMIA_NO_REGENERATIVA,42,0.793085,42,1,0,0.976744,1.000000,0.988235,0.985340
4,v3,PATRON_HEMOLISIS_MCHC,48,0.772114,44,2,4,0.956522,0.916667,0.936170,0.992356
5,v3,PATRON_POLICITEMIA,32,0.995109,28,0,4,1.000000,0.875000,0.933333,1.000000
6,v3,PATRON_ANEMIA_REGENERATIVA,6,0.379511,5,9,1,0.357143,0.833333,0.500000,0.885965
7,v4_runtime,QC_REQUIERE_FROTIS,136,0.300000,93,26,43,0.781513,0.683824,0.729412,0.845496
8,v4_runtime,PATRON_INFLAMATORIO,198,0.641006,197,4,1,0.980100,0.994949,0.987469,0.993141
9,v4_runtime,PATRON_LEUCOGRAMA_ESTRES,172,0.300000,171,11,1,0.939560,0.994186,0.966102,0.983335


{'recall_loss_le_2pp': False, 'f1_not_lower_per_label': False, 'fp_lower_than_v4': False, 'fn_lower_than_v3': True, 'promote_candidate': False}
Guardado: /home/edwinb/tesis/hemogramas-proyectoICC/outputs/comparacion_modelos.csv


In [12]:
# Promoción transaccional del bundle runtime.
import hashlib
import shutil
from datetime import datetime, timezone

selected_thresholds = candidate_thresholds if promote_candidate else runtime_thresholds
selected_calibrators = candidate_calibrators if promote_candidate else {}
selected_metrics = df_candidate_test if promote_candidate else df_v4_test
selection = 'v4_balanced_candidate' if promote_candidate else 'v4_runtime_retained'

thresholds_candidate_doc = {
    'version': '4.1.0-candidate',
    'source_split': 'val_oof',
    'model': 'xgb_v4_results_augmented',
    'promoted': bool(promote_candidate),
    'criteria': {
        'max_recall_loss_per_label': 0.02,
        'f1_not_lower_per_label': True,
        'fp_lower_than_v4_runtime': True,
        'fn_lower_than_v3': True,
    },
    'thresholds': candidate_thresholds,
}
with open(DATA_PROC / 'decision_thresholds_v4.json', 'w') as f:
    json.dump(thresholds_candidate_doc, f, indent=2)

shutil.copyfile(MODELS / 'best_model_v4.pkl', MODELS / 'best_model_v2.pkl')
with open(MODELS / 'calibrators_v2.pkl', 'wb') as f:
    pickle.dump(selected_calibrators, f)

runtime_thresholds_doc = {
    'version': '4.0.0' if not promote_candidate else '4.1.0',
    'source_split': 'val_oof' if promote_candidate else 'val',
    'model': 'xgb_v4_results_augmented',
    'selection': selection,
    'note': (
        'Candidato recalibrado promovido con restricciones FP/FN.'
        if promote_candidate else
        'Se conserva v4: el candidato recalibrado no cumplió todas las restricciones FP/FN.'
    ),
    'thresholds': selected_thresholds,
}
with open(DATA_PROC / 'decision_thresholds_v2.json', 'w') as f:
    json.dump(runtime_thresholds_doc, f, indent=2)

metadata_path = MODELS / 'model_metadata_v2.json'
with open(metadata_path) as f:
    metadata = json.load(f)
metadata.update({
    'version': runtime_thresholds_doc['version'],
    'model_type': 'xgb_v4_results_augmented',
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'train_n': int(len(df_train_v4)),
    'train_base_n': int(len(train_base)),
    'train_mythic_n': int(len(results_aug)),
    'val_n': int(len(val_df)),
    'test_n': int(len(test_df)),
    'prauc_macro': float(selected_metrics.prauc.mean()),
    'calibration': 'sigmoid_logit' if promote_candidate else 'none_candidate_rejected',
    'threshold_selection': selection,
    'test_metrics': {
        'prauc_macro': float(selected_metrics.prauc.mean()),
        'brier_macro': float(selected_metrics.brier.mean()),
        'f1_macro': float(selected_metrics.f1.mean()),
        'recall_macro': float(selected_metrics.rec.mean()),
        'tp': int(selected_metrics.tp.sum()),
        'fp': int(selected_metrics.fp.sum()),
        'fn': int(selected_metrics.fn.sum()),
        'tn': int(selected_metrics.tn.sum()),
    },
    'limitations': [
        'Las 1213 etiquetas MYTHIC se derivan de flags y proxies, no de anotación clínica independiente.',
        'PATRON_ANEMIA_REGENERATIVA es exploratoria: 6 positivos en test y 0 positivos MYTHIC nuevos.',
        'POLICITEMIA y HEMOLISIS_MCHC tienen riesgo de circularidad por HCT y MCHC.',
    ],
})
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

final_state_path = PROJECT / 'outputs' / 'final_system_state.json'
final_state = {
    'version': runtime_thresholds_doc['version'],
    'status': 'READY_FOR_PRODUCTION_WITH_LIMITATIONS',
    'model_file': 'models/best_model_v4.pkl',
    'runtime_model_file': 'models/best_model_v2.pkl',
    'runtime_thresholds_file': 'data/processed/decision_thresholds_v2.json',
    'threshold_selection': selection,
    'n_official_labels': len(OFFICIAL_LABELS),
    'test_metrics': metadata['test_metrics'],
    'rare_label_policy': {
        'label': 'PATRON_ANEMIA_REGENERATIVA',
        'status': 'exploratory_low_support',
        'n_positive_test': 6,
        'threshold_adjusted': False,
    },
    'limitations': metadata['limitations'],
}
with open(final_state_path, 'w') as f:
    json.dump(final_state, f, indent=2, ensure_ascii=False)

manifest_path = DATA_PROC / 'artifact_manifest_v2.json'
with open(manifest_path) as f:
    manifest = json.load(f)
manifest['version'] = runtime_thresholds_doc['version']
manifest['generated_at'] = datetime.now(timezone.utc).isoformat()
for section in ('runtime_artifacts', 'support_artifacts'):
    for entry in manifest.get(section, []):
        artifact = PROJECT / entry['path']
        if artifact.exists():
            entry['sha256'] = hashlib.sha256(artifact.read_bytes()).hexdigest()
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Bundle runtime sincronizado: {selection}')
print(f'Modelo: {MODELS / "best_model_v2.pkl"}')
print(f'Umbrales: {DATA_PROC / "decision_thresholds_v2.json"}')
print(f'Metadata: {metadata_path}')
print(f'Estado final: {final_state_path}')
print(f'Manifiesto: {manifest_path}')

Bundle runtime sincronizado: v4_runtime_retained
Modelo: /home/edwinb/tesis/hemogramas-proyectoICC/models/best_model_v2.pkl
Umbrales: /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/decision_thresholds_v2.json
Metadata: /home/edwinb/tesis/hemogramas-proyectoICC/models/model_metadata_v2.json
Estado final: /home/edwinb/tesis/hemogramas-proyectoICC/outputs/final_system_state.json
Manifiesto: /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/artifact_manifest_v2.json
